# 06 — Évaluation de l'agent

Deux évaluations, sur le gold set Chinook (`src/text_to_sql_mcp/evaluation/gold_chinook.toml`) :

1. **Évaluation 1 — justesse, abstention, latence, coût.** Les 44 questions, une fois chacune,
   pour chaque modèle.
2. **Évaluation 2 — fiabilité (pass^k).** Les questions difficiles, *k* fois chacune : l'agent
   réussit-il **à chaque fois** ?

Le code de l'évaluation est dans le package (`text_to_sql_mcp.evaluation`) et testé ; ce
notebook ne fait que lancer, relire et afficher. Chaque évaluation est écrite au fil de l'eau
dans `evals/resultats/*.jsonl` : pour réafficher des résultats sans relancer l'agent, renseigne
`RECHARGER_EVAL_1` / `RECHARGER_EVAL_2` dans la cellule de configuration.

**Répétition à blanc.** Avec `SIMULATION = True`, un oracle remplace l'API : il joue la
requête de référence (en se trompant volontairement sur 15 % des questions). Tout le reste est
réel (boucle, serveur MCP, base, notation) et rien n'est facturé. À lancer une première fois
pour vérifier la chaîne et les graphiques ; les scores obtenus ne mesurent rien du modèle.

**Prérequis :** base démarrée, `ANTHROPIC_API_KEY` dans le `.env`, kernel `.venv`.

## 0. Configuration

In [31]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # racine du repo : .env, et dossier evals/

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from text_to_sql_mcp.evaluation import (
    OracleAnthropic,
    compute_references,
    evaluate,
    load_gold_set,
    load_records,
    new_run_id,
    pass_at_k,
    pass_hat_k,
    wilson_interval,
)

HAIKU, SONNET = "claude-haiku-5-5", "claude-sonnet-5-5"

SIMULATION = False  # True : répétition à blanc, sans API ni coût
MODELES = [HAIKU, SONNET]
PROMPT = "v1"  # v1 : consigne d'abstention avec le marqueur [SANS_REPONSE]
CONCURRENCE = 4  # agents en parallèle
K = 5  # évaluation 2 : nombre de passages par question difficile
DOSSIER = Path("evals/resultats")

# Pour réafficher sans relancer l'agent : fichiers JSONL d'une évaluation précédente.
RECHARGER_EVAL_1: list[Path] = []
RECHARGER_EVAL_2: list[Path] = []

Thème des graphiques : une couleur fixe par modèle, fond clair, grille discrète. Les verdicts juste / faux portent toujours un
symbole (✓ / ✗) en plus de leur couleur.

In [32]:
NOMS = {HAIKU: "Haiku 5.5", SONNET: "Sonnet 5.5"}
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
COULEURS = {NOMS.get(m, m): PALETTE[i % len(PALETTE)] for i, m in enumerate(MODELES)}
JUSTE, FAUX = "#0ca30c", "#d03b3b"
SURFACE, ENCRE, ENCRE_2, GRILLE, AXE = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"
BLEUS = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

axe = {"gridcolor": GRILLE, "linecolor": AXE, "zeroline": False, "ticks": "", "title_standoff": 8}
pio.templates["projet"] = go.layout.Template(
    layout={
        "font": {
            "family": "system-ui, -apple-system, Segoe UI, sans-serif",
            "color": ENCRE,
            "size": 13,
        },
        "paper_bgcolor": SURFACE,
        "plot_bgcolor": SURFACE,
        "colorway": PALETTE,
        "xaxis": axe,
        "yaxis": axe,
        "legend": {"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0, "title_text": ""},
        "margin": {"l": 60, "r": 20, "t": 70, "b": 50},
        "barcornerradius": 4,
        "hoverlabel": {"bgcolor": "white", "font_color": ENCRE},
    }
)
pio.templates.default = "plotly_white+projet"
pd.set_option("display.max_colwidth", 120)


def nom(modele: str) -> str:
    return NOMS.get(modele, modele)


def tableau(enregistrements) -> pd.DataFrame:
    """Un enregistrement par ligne, sans le détail complet de l'agent."""
    df = pd.DataFrame([e.model_dump(exclude={"agent"}) for e in enregistrements])
    df["modele"] = df["model"].map(nom)
    return df


def progression(total: int, modele: str):
    """Affiche l'avancement toutes les 10 réponses."""
    compte = {"n": 0, "justes": 0}

    def suivre(e) -> None:
        compte["n"] += 1
        compte["justes"] += e.correct
        if compte["n"] % 10 == 0 or compte["n"] == total:
            print(f"  {nom(modele)} : {compte['n']}/{total} réponses, {compte['justes']} justes")

    return suivre


def client(modele: str, questions):
    """Oracle simulé (SIMULATION = True), sinon None : le harnais crée le vrai client."""
    if not SIMULATION:
        return None
    return OracleAnthropic(questions, error_rate=0.15, seed=MODELES.index(modele))

## 1. Le gold set

44 questions reprises du dataset *Chinook gold* du projet
[nl2sql](https://github.com/nadeem4/nl2sql) (licence MIT), traduites en français et adaptées à
notre base (PostgreSQL, snake_case, dates décalées de +12 ans), plus une question maison.
Chaque traduction a été vérifiée : elle redonne exactement le résultat publié par la source.

- **39 questions répondables**, avec une ou plusieurs requêtes de référence (plusieurs formes de
  réponse justes : nom complet en une ou deux colonnes, mois en texte ou en date, ex aequo…) ;
- **5 questions sans réponse** : la base ne contient pas l'information (météo, écoutes Spotify,
  marges, avis clients, salaires) ; l'agent doit s'abstenir.

In [33]:
questions = load_gold_set()
gold = pd.DataFrame(
    [
        {
            "id": q.id,
            "question": q.question,
            "difficulte": q.difficulty,
            "repondable": q.answerable,
            "ordre": q.ordered,
            "references": len(q.references),
        }
        for q in questions
    ]
)
ordre_difficulte = ["facile", "moyen", "difficile"]
compte = (
    gold.assign(type=gold["repondable"].map({True: "Répondable", False: "Sans réponse"}))
    .groupby(["difficulte", "type"])
    .size()
    .reset_index(name="questions")
)
fig = go.Figure()
for i, (type_, groupe) in enumerate(compte.groupby("type")):
    groupe = groupe.set_index("difficulte").reindex(ordre_difficulte).fillna(0)
    fig.add_bar(
        name=type_,
        x=ordre_difficulte,
        y=groupe["questions"],
        marker_color=PALETTE[i],
        marker_line_color=SURFACE,
        marker_line_width=2,
        text=[str(int(v)) if v else "" for v in groupe["questions"]],
        textposition="outside",
        hovertemplate="%{x} : %{y} questions<extra>" + type_ + "</extra>",
    )
fig.update_layout(
    title="Questions du gold set par difficulté",
    barmode="group",
    height=360,
    yaxis_title="Questions",
)
fig.show()
gold.head(8)

,id,question,difficulte,repondable,ordre,references
0,chinook_001,Combien avons-nous de clients par pays ?,facile,True,False,1
1,chinook_002,Quels sont les 5 clients qui ont le plus dépensé au total ?,moyen,True,False,2
2,chinook_003,Quel artiste a le plus d'albums ?,facile,True,True,1
3,chinook_004,Quel est le chiffre d'affaires total par année ?,moyen,True,False,2
4,chinook_005,Quel genre vend le plus de pistes ?,moyen,True,True,1
5,chinook_006,Quel est le montant moyen d'une facture par pays de facturation ?,facile,True,False,1
6,chinook_007,Quels employés suivent le plus de clients ?,moyen,True,True,4
7,chinook_008,Quelle est la piste la plus longue de chaque genre ?,difficile,True,False,1


Les requêtes de référence sont exécutées une fois, avec le **même rôle lecture seule** que
l'agent (elles passent aussi par les garde-fous). Le plafond de lignes de l'agent est 100 :
aucune référence ne doit le dépasser.

In [34]:
questions[0:2]

[GoldQuestion(id='chinook_001', question='Combien avons-nous de clients par pays ?', question_en='How many customers do we have, by country?', difficulty='facile', tags=('aggregation',), ordered=False, answerable=True, references=('SELECT country, COUNT(*) AS nb_clients FROM customer GROUP BY country\n',)),
 GoldQuestion(id='chinook_002', question='Quels sont les 5 clients qui ont le plus dépensé au total ?', question_en='Who are the top 5 customers by total spend?', difficulty='moyen', tags=('join', 'aggregation', 'order-limit', 'paraphrase'), ordered=False, answerable=True, references=('SELECT c.first_name, c.last_name\nFROM customer c JOIN invoice i ON i.customer_id = c.customer_id\nGROUP BY c.customer_id ORDER BY SUM(i.total) DESC, c.customer_id LIMIT 5\n', "SELECT c.first_name || ' ' || c.last_name AS client\nFROM customer c JOIN invoice i ON i.customer_id = c.customer_id\nGROUP BY c.customer_id ORDER BY SUM(i.total) DESC, c.customer_id LIMIT 5\n"))]

In [35]:
references = compute_references(questions)
tailles = [len(lignes) for refs in references.values() for lignes in refs]
print(f"{len(tailles)} requêtes de référence exécutées ; la plus longue : {max(tailles)} lignes")

61 requêtes de référence exécutées ; la plus longue : 25 lignes


## 2. Évaluation 1 — justesse, abstention, latence, coût

Chaque question est posée une fois à chaque modèle, avec le prompt `v1`. Règles de notation
(module `evaluation.comparison`) :

| Cas | Juste si… |
|---|---|
| Question répondable | le résultat de la dernière requête réussie de l'agent égale celui d'une référence : même nombre de lignes ; chaque colonne de la référence présente (colonnes en plus tolérées) ; ordre comparé seulement si la question l'implique ; `"10"`, `10` et `10.00` égaux |
| Question sans réponse | la réponse commence par `[SANS_REPONSE]` |

Le marqueur sur une question répondable est un **faux refus**. Latence : durée totale de
l'agent (tous les tours, outils compris). Coût : tokens d'entrée et de sortie (réflexion
comprise) au tarif public de chaque modèle.

In [36]:
RECHARGER_EVAL_1

[]

In [37]:
SIMULATION

False

In [38]:
if RECHARGER_EVAL_1:
    eval1 = load_records(*RECHARGER_EVAL_1)
else:
    eval1 = []
    for modele in MODELES:
        run_id = new_run_id(f"eval1_{modele}_{PROMPT}" + ("_simulation" if SIMULATION else ""))
        fichier = DOSSIER / f"{run_id}.jsonl"
        print(f"{nom(modele)} -> {fichier}")
        eval1 += await evaluate(
            questions,
            model=modele,
            prompt_name=PROMPT,
            references=references,
            concurrency=CONCURRENCE,
            output=fichier,
            run_id=run_id,
            anthropic=client(modele, questions),
            on_record=progression(len(questions), modele),
        )
df1 = tableau(eval1)

Haiku 5.5 -> evals/resultats/20261009T140140Z_eval1_claude-haiku-5-5_v1.jsonl
  Haiku 5.5 : 10/44 réponses, 6 justes
  Haiku 5.5 : 20/44 réponses, 14 justes
  Haiku 5.5 : 30/44 réponses, 24 justes
  Haiku 5.5 : 40/44 réponses, 32 justes
  Haiku 5.5 : 44/44 réponses, 36 justes
Sonnet 5.5 -> evals/resultats/20261009T140239Z_eval1_claude-sonnet-5-5_v1.jsonl
  Sonnet 5.5 : 10/44 réponses, 8 justes
  Sonnet 5.5 : 20/44 réponses, 15 justes
  Sonnet 5.5 : 30/44 réponses, 23 justes
  Sonnet 5.5 : 40/44 réponses, 31 justes
  Sonnet 5.5 : 44/44 réponses, 35 justes


### 2.1 Synthèse

L'intervalle de confiance (95 %, méthode de Wilson) rappelle la taille de l'échantillon : avec
39 questions, une question pèse 2,6 points, et deux modèles dont les intervalles se recouvrent
largement ne sont pas départagés par ce seul passage.

In [39]:
def synthese(df: pd.DataFrame) -> pd.DataFrame:
    lignes = []
    for modele, g in df.groupby("modele", sort=False):
        rep, sans = g[g["answerable"]], g[~g["answerable"]]
        bas, haut = wilson_interval(int(rep["correct"].sum()), len(rep))
        lignes.append(
            {
                "modèle": modele,
                "execution accuracy": rep["correct"].mean(),
                "IC 95 % bas": bas,
                "IC 95 % haut": haut,
                "abstentions correctes": f"{int(sans['correct'].sum())}/{len(sans)}",
                "faux refus": int((g["reason"] == "faux_refus").sum()),
                "erreurs agent": int((g["reason"] == "erreur_agent").sum()),
                "latence médiane (s)": g["duration_s"].median(),
                "latence p95 (s)": g["duration_s"].quantile(0.95),
                "tours moyens": g["turns"].mean(),
                "coût moyen / question ($)": g["cost_usd"].mean(),
                "coût total ($)": g["cost_usd"].sum(),
            }
        )
    return pd.DataFrame(lignes).set_index("modèle")


resume1 = synthese(df1)
resume1.style.format(
    {
        "execution accuracy": "{:.1%}",
        "IC 95 % bas": "{:.1%}",
        "IC 95 % haut": "{:.1%}",
        "latence médiane (s)": "{:.1f}",
        "latence p95 (s)": "{:.1f}",
        "tours moyens": "{:.1f}",
        "coût moyen / question ($)": "{:.4f}",
        "coût total ($)": "{:.3f}",
    }
)

,execution accuracy,IC 95 % bas,IC 95 % haut,abstentions correctes,faux refus,erreurs agent,latence médiane (s),latence p95 (s),tours moyens,coût moyen / question ($),coût total ($)
modèle,,,,,,,,,,,
Haiku 5.5,79.5%,64.5%,89.2%,5/5,0,0,4.8,6.4,3.9,0.0011,0.050
Sonnet 5.5,76.9%,61.7%,87.4%,5/5,0,0,5.7,9.6,3.7,0.0208,0.914


In [40]:
fig = go.Figure()
for modele, ligne in resume1.iterrows():
    ex = ligne["execution accuracy"]
    fig.add_bar(
        name=modele,
        x=[modele],
        y=[ex],
        marker_color=COULEURS.get(modele),
        error_y={
            "type": "data",
            "symmetric": False,
            "array": [ligne["IC 95 % haut"] - ex],
            "arrayminus": [ex - ligne["IC 95 % bas"]],
            "color": ENCRE_2,
            "thickness": 1.5,
        },
        text=[f"{ex:.0%}"],
        textposition="inside",
        insidetextanchor="start",
        textfont_color="white",
        width=0.5,
        hovertemplate=f"{modele}<br>%{{y:.1%}} (IC 95 % : {ligne['IC 95 % bas']:.0%}"
        f"–{ligne['IC 95 % haut']:.0%})<extra></extra>",
    )
fig.update_layout(
    title="Execution accuracy sur les 39 questions répondables (IC 95 %)",
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1.05],
    showlegend=False,
    height=380,
)
fig.show()

### 2.2 Justesse par difficulté

In [41]:
par_difficulte = (
    df1[df1["answerable"]]
    .groupby(["modele", "difficulty"])["correct"]
    .agg(["mean", "sum", "count"])
    .reset_index()
)
fig = go.Figure()
for modele in resume1.index:
    g = (
        par_difficulte[par_difficulte["modele"] == modele]
        .set_index("difficulty")
        .reindex(ordre_difficulte)
    )
    fig.add_bar(
        name=modele,
        x=ordre_difficulte,
        y=g["mean"],
        marker_color=COULEURS.get(modele),
        marker_line_color=SURFACE,
        marker_line_width=2,
        customdata=g[["sum", "count"]].to_numpy(),
        hovertemplate="%{x} : %{y:.0%} (%{customdata[0]}/%{customdata[1]})<extra>"
        + modele
        + "</extra>",
    )
fig.update_layout(
    title="Execution accuracy par difficulté",
    barmode="group",
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1.05],
    height=380,
)
fig.show()

### 2.3 Pourquoi l'agent échoue

Motifs d'échec : `nb_lignes` (trop ou trop peu de lignes), `colonnes` (il manque une colonne
de la réponse), `valeurs`, `ordre` (bonnes lignes, mauvais ordre), `aucune_requete`,
`tronque` (résultat plafonné), `faux_refus`, `reponse_inventee` (pas d'abstention sur une
question sans réponse), `erreur_agent` (API ou serveur).

In [42]:
echecs = df1[~df1["correct"]]
if echecs.empty:
    print("Aucun échec.")
else:
    motifs = echecs.groupby(["reason", "modele"]).size().unstack(fill_value=0)
    motifs = motifs.loc[motifs.sum(axis=1).sort_values().index]
    fig = go.Figure()
    for modele in reversed(resume1.index):  # barres horizontales : ordre inversé à l'écran
        if modele in motifs:
            fig.add_bar(
                name=modele,
                y=motifs.index,
                x=motifs[modele],
                orientation="h",
                marker_color=COULEURS.get(modele),
                marker_line_color=SURFACE,
                marker_line_width=2,
                hovertemplate="%{y} : %{x}<extra>" + modele + "</extra>",
            )
    fig.update_layout(
        title="Motifs d'échec (nombre de questions)",
        barmode="group",
        legend_traceorder="reversed",
        xaxis_title="Questions",
        xaxis_dtick=1,
        height=120 + 45 * len(motifs),
    )
    fig.show()

### 2.4 Question par question

✓ juste, ✗ faux ; le survol donne le motif. Les questions sur lesquelles les modèles divergent
sont listées sous la carte : c'est la comparaison la plus informative, car elle porte sur les
mêmes questions.

In [43]:
carte = df1.pivot_table(index="question_id", columns="modele", values="correct", aggfunc="first")
carte = carte[[m for m in resume1.index if m in carte]]
motif = df1.pivot_table(index="question_id", columns="modele", values="reason", aggfunc="first")
textes = df1.drop_duplicates("question_id").set_index("question_id")["question"]
survol = [
    [f"{qid} — {textes[qid]}<br>{m} : {motif.loc[qid, m]}" for m in carte.columns]
    for qid in carte.index
]
fig = go.Figure(
    go.Heatmap(
        z=carte.astype(int).to_numpy(),
        x=list(carte.columns),
        y=list(carte.index),
        colorscale=[[0, FAUX], [1, JUSTE]],
        zmin=0,
        zmax=1,
        showscale=False,
        xgap=2,
        ygap=2,
        text=[["✓" if v else "✗" for v in ligne] for ligne in carte.to_numpy()],
        texttemplate="%{text}",
        textfont={"color": "white", "size": 14},
        hovertext=survol,
        hovertemplate="%{hovertext}<extra></extra>",
    )
)
fig.update_layout(
    title="Verdict par question",
    height=24 * len(carte) + 120,
    width=520,
    yaxis_autorange="reversed",
    xaxis_side="top",
    margin={"t": 100},
)
fig.show()

divergences = carte[carte.nunique(axis=1) > 1]
print(f"{len(divergences)} question(s) où les modèles divergent")
divergences.join(textes).replace({True: "✓", False: "✗"})

5 question(s) où les modèles divergent


,Haiku 5.5,Sonnet 5.5,question
question_id,,,
chinook_007,✗,✓,Quels employés suivent le plus de clients ?
chinook_009,✗,✓,Quels clients ont acheté des pistes de jazz mais jamais de rock ?
chinook_017,✓,✗,Quel musicien a sorti plus d'albums que tous les autres dans le catalogue ?
chinook_029,✓,✗,Quel mois de 2023 a enregistré le plus gros chiffre d'affaires ?
chinook_030,✓,✗,Quels sont les 3 pays qui comptent le plus de clients ?


### 2.5 Détail des échecs

Le SQL exécuté par l'agent et le début de sa réponse : de quoi comprendre chaque échec (et
vérifier qu'il ne vient pas d'une règle de notation trop stricte).

In [44]:
(
    df1[~df1["correct"]]
    .assign(reponse=lambda d: d["answer"].str.slice(0, 200))[
        ["question_id", "modele", "reason", "question", "executed_sql", "reponse"]
    ]
    .sort_values(["question_id", "modele"])
)

,question_id,modele,reason,question,executed_sql,reponse
2,chinook_003,Haiku 5.5,nb_lignes,Quel artiste a le plus d'albums ?,"SELECT ar.artist_id, ar.name, COUNT(al.album_id) AS nb_albums FROM artist AS ar JOIN album AS al ON al.artist_id = a...","Iron Maiden est l'artiste qui a le plus d'albums, avec 21 albums, devant Led Zeppelin (14) et Deep Purple (11).\n\n`..."
46,chinook_003,Sonnet 5.5,nb_lignes,Quel artiste a le plus d'albums ?,"SELECT ar.name, COUNT(*) AS nb_albums FROM album AS al JOIN artist AS ar ON ar.artist_id = al.artist_id GROUP BY ar....","L'artiste qui a le plus d'albums est **Iron Maiden**, avec 21 albums. Il devance Led Zeppelin (14) et Deep Purple (1..."
4,chinook_005,Haiku 5.5,nb_lignes,Quel genre vend le plus de pistes ?,"SELECT g.name AS genre, SUM(il.quantity) AS pistes_vendues FROM invoice_line AS il JOIN track AS t ON t.track_id = i...","Le genre qui vend le plus de pistes est le Rock, avec 835 pistes vendues, loin devant le Latin (386) et le Metal (26..."
48,chinook_005,Sonnet 5.5,nb_lignes,Quel genre vend le plus de pistes ?,"SELECT g.name, SUM(il.quantity) AS pistes_vendues FROM invoice_line AS il JOIN track AS t ON t.track_id = il.track_i...","Le genre qui vend le plus de pistes est le **Rock**, avec 835 pistes vendues. Il devance le Latin (386) et le Metal ..."
6,chinook_007,Haiku 5.5,nb_lignes,Quels employés suivent le plus de clients ?,"SELECT e.employee_id, e.first_name, e.last_name, e.title, COUNT(c.customer_id) AS nb_clients FROM employee AS e LEFT...","Jane Peacock est l'employée qui suit le plus de clients, avec 21 clients, suivie de Margaret Park (20) et Steve John..."
8,chinook_009,Haiku 5.5,nb_lignes,Quels clients ont acheté des pistes de jazz mais jamais de rock ?,"SELECT COUNT(DISTINCT i.customer_id) AS clients_jazz, COUNT(DISTINCT i.customer_id) FILTER(WHERE EXISTS(SELECT 1 FRO...",Aucun client n'a acheté du jazz sans avoir aussi acheté du rock : les 32 clients ayant acheté du jazz ont tous achet...
11,chinook_012,Haiku 5.5,nb_lignes,Qui sont nos meilleurs clients en dépense totale ?,"SELECT c.customer_id, c.first_name, c.last_name, c.country, ROUND(SUM(i.total), 2) AS total_depense, COUNT(i.invoice...","Les 10 clients qui ont le plus dépensé sont en tête Helena Holý (Czech Republic, 49,62 €), suivie de Richard Cunning..."
55,chinook_012,Sonnet 5.5,nb_lignes,Qui sont nos meilleurs clients en dépense totale ?,"SELECT c.customer_id, c.first_name, c.last_name, c.country, SUM(i.total) AS total_depense FROM customer AS c JOIN in...","Helena Holý (République tchèque) est notre meilleure cliente, avec 49,62 de dépenses totales. Elle est suivie de Ric..."
15,chinook_016,Haiku 5.5,nb_lignes,Nomme l'artiste qui a le plus grand nombre d'albums.,"SELECT ar.name, COUNT(al.album_id) AS nb_albums FROM artist AS ar JOIN album AS al ON al.artist_id = ar.artist_id GR...","L'artiste qui possède le plus d'albums est Iron Maiden, avec 21 albums.\n\n```sql\nSELECT ar.name, COUNT(al.album_id..."
59,chinook_016,Sonnet 5.5,nb_lignes,Nomme l'artiste qui a le plus grand nombre d'albums.,"SELECT ar.name, COUNT(*) AS nb_albums FROM album AS al JOIN artist AS ar ON ar.artist_id = al.artist_id GROUP BY ar....","L'artiste qui a le plus d'albums est **Iron Maiden**, avec 21 albums. Led Zeppelin vient ensuite avec 14 albums.\n\n..."


### 2.6 Latence et coût

Latence : chaque point est une question (survol : laquelle). Coût : le compromis entre justesse
et prix, un point par modèle.

In [45]:
fig = go.Figure()
for modele in resume1.index:
    g = df1[df1["modele"] == modele]
    fig.add_box(
        name=modele,
        y=g["duration_s"],
        marker_color=COULEURS.get(modele),
        boxpoints="all",
        jitter=0.4,
        pointpos=0,
        marker_size=8,
        line_width=2,
        text=g["question_id"],
        hovertemplate="%{text} : %{y:.1f} s<extra>" + modele + "</extra>",
    )
fig.update_layout(
    title="Latence par question (secondes)", yaxis_title="Secondes", showlegend=False, height=420
)
fig.show()

In [47]:
fig = go.Figure()
for modele, ligne in resume1.iterrows():
    fig.add_scatter(
        x=[ligne["coût moyen / question ($)"]],
        y=[ligne["execution accuracy"]],
        mode="markers+text",
        name=modele,
        text=[modele],
        textposition="middle right",
        marker={"size": 16, "color": COULEURS.get(modele), "line": {"color": SURFACE, "width": 2}},
        hovertemplate=f"{modele}<br>%{{x:$.4f}} par question<br>%{{y:.1%}}<extra></extra>",
    )
cout_max = resume1["coût moyen / question ($)"].max()
fig.update_layout(
    title="Justesse et coût moyen par question",
    showlegend=False,
    height=380,
    xaxis_title="Coût moyen par question ($)",
    xaxis_tickformat="$.4f",
    xaxis_range=[-0.05 * cout_max, cout_max * 1.5] if cout_max > 0 else [0, 1],
    yaxis_title="Execution accuracy",
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1.05],
)
fig.show()

## 3. Évaluation 2 — fiabilité sur les questions difficiles (pass^k)

Les questions difficiles du gold set, posées **K fois** chacune. Pour une question réussie
*c* fois sur *n* :

- **pass^k** = probabilité que *k* essais réussissent **tous** = C(c, k) / C(n, k) :
  ce sur quoi on peut compter ;
- **pass@k** = probabilité qu'**au moins un** essai sur *k* réussisse : ce que l'agent sait
  faire.

Moyennées sur les questions, les deux courbes partent du même point (k = 1, la justesse
moyenne) et s'écartent avec k : l'écart mesure l'instabilité de l'agent.

In [48]:
questions_difficiles = [q for q in questions if q.difficulty == "difficile"]
print(f"{len(questions_difficiles)} questions difficiles x {K} passages x {len(MODELES)} modèles")
for q in questions_difficiles:
    print(f"  {q.id} — {q.question}")

6 questions difficiles x 5 passages x 2 modèles
  chinook_008 — Quelle est la piste la plus longue de chaque genre ?
  chinook_009 — Quels clients ont acheté des pistes de jazz mais jamais de rock ?
  chinook_011 — Quelles playlists contiennent des pistes de plus de trois genres ?
  chinook_032 — Quel artiste a généré le plus de chiffre d'affaires ?
  chinook_034 — Combien de clients différents ont acheté au moins une piste d'AC/DC ?
  chinook_037 — Quel agent du support commercial a rapporté le plus de chiffre d'affaires grâce à ses clients ?


In [49]:
RECHARGER_EVAL_2

[]

In [50]:
SIMULATION

False

In [ ]:
if RECHARGER_EVAL_2:
    eval2 = load_records(*RECHARGER_EVAL_2)
else:
    eval2 = []
    for modele in MODELES:
        run_id = new_run_id(f"eval2_{modele}_{PROMPT}_k{K}" + ("_simulation" if SIMULATION else ""))
        fichier = DOSSIER / f"{run_id}.jsonl"
        print(f"{nom(modele)} -> {fichier}")
        eval2 += await evaluate(
            questions_difficiles,
            model=modele,
            prompt_name=PROMPT,
            references=references,
            repetitions=K,  # K=5 ici
            concurrency=CONCURRENCE,
            output=fichier,
            run_id=run_id,
            anthropic=client(
                modele, questions_difficiles
            ),  # None si pas en simulation, la fonction crée un client elle même avec le .env  # noqa: E501
            on_record=progression(len(questions_difficiles) * K, modele),
        )
df2 = tableau(eval2)

Haiku 5.5 -> evals/resultats/20261009T141124Z_eval2_claude-haiku-5-5_v1_k5.jsonl
  Haiku 5.5 : 10/30 réponses, 6 justes
  Haiku 5.5 : 20/30 réponses, 10 justes
  Haiku 5.5 : 30/30 réponses, 15 justes
Sonnet 5.5 -> evals/resultats/20261009T141216Z_eval2_claude-sonnet-5-5_v1_k5.jsonl
  Sonnet 5.5 : 10/30 réponses, 10 justes
  Sonnet 5.5 : 20/30 réponses, 15 justes
  Sonnet 5.5 : 30/30 réponses, 20 justes


In [52]:
reussites = df2.groupby(["modele", "question_id"])["correct"].agg(c="sum", n="count").reset_index()
n_min = int(reussites["n"].min())
courbes = []
for modele, g in reussites.groupby("modele", sort=False):
    for k in range(1, n_min + 1):
        courbes.append(
            {
                "modele": modele,
                "k": k,
                "pass^k": sum(pass_hat_k(int(r.n), int(r.c), k) for r in g.itertuples()) / len(g),
                "pass@k": sum(pass_at_k(int(r.n), int(r.c), k) for r in g.itertuples()) / len(g),
            }
        )
courbes = pd.DataFrame(courbes)

fig = go.Figure()
for modele in [m for m in resume1.index if m in set(courbes["modele"])] or courbes[
    "modele"
].unique():
    g = courbes[courbes["modele"] == modele]
    for metrique, tirets in (("pass@k", "dot"), ("pass^k", "solid")):
        fig.add_scatter(
            x=g["k"],
            y=g[metrique],
            name=f"{modele} — {metrique}",
            mode="lines+markers",
            line={"color": COULEURS.get(modele), "width": 2, "dash": tirets},
            marker_size=8,
            hovertemplate=f"{modele}<br>{metrique} (k=%{{x}}) : %{{y:.0%}}<extra></extra>",
        )
fig.update_layout(
    title=f"pass^k et pass@k sur {len(questions_difficiles)} questions difficiles",
    xaxis_title="k",
    xaxis_dtick=1,
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1.05],
    height=420,
)
fig.show()
courbes.pivot(index="k", columns="modele", values="pass^k").style.format("{:.0%}")

modele,Haiku 5.5,Sonnet 5.5
k,,
1,50%,67%
2,50%,67%
3,50%,67%
4,50%,67%
5,50%,67%


Réussites par question : une question à 5/5 est fiable ; une question à 2/5 ou 3/5 est le
signe d'une instabilité (formulation ambiguë, ex aequo, piège de jointure…) à examiner dans les
réponses ci-dessous.

In [53]:
grille = reussites.pivot(index="question_id", columns="modele", values="c")
grille = grille[[m for m in resume1.index if m in grille] or list(grille.columns)]
n_par_case = reussites.pivot(index="question_id", columns="modele", values="n")[grille.columns]
textes2 = df2.drop_duplicates("question_id").set_index("question_id")["question"]
fig = go.Figure(
    go.Heatmap(
        z=grille.to_numpy(),
        x=list(grille.columns),
        y=list(grille.index),
        zmin=0,
        zmax=n_min,
        colorscale=[[i / (len(BLEUS) - 1), c] for i, c in enumerate(BLEUS)],
        xgap=2,
        ygap=2,
        text=[
            [f"{int(c)}/{int(n)}" for c, n in zip(lc, ln, strict=True)]
            for lc, ln in zip(grille.to_numpy(), n_par_case.to_numpy(), strict=True)
        ],
        texttemplate="%{text}",
        colorbar={"title": "réussites", "dtick": 1},
        hovertext=[[f"{qid} — {textes2[qid]}" for _ in grille.columns] for qid in grille.index],
        hovertemplate="%{hovertext}<br>%{x} : %{text}<extra></extra>",
    )
)
fig.update_layout(
    title=f"Réussites sur {n_min} passages",
    height=40 * len(grille) + 140,
    width=560,
    yaxis_autorange="reversed",
    xaxis_side="top",
    margin={"t": 100},
)
fig.show()

(
    df2[~df2["correct"]]
    .groupby(["question_id", "modele", "reason"])
    .size()
    .rename("échecs")
    .reset_index()
)

,question_id,modele,reason,échecs
0,chinook_009,Haiku 5.5,nb_lignes,5
1,chinook_032,Haiku 5.5,nb_lignes,5
2,chinook_032,Sonnet 5.5,nb_lignes,5
3,chinook_037,Haiku 5.5,nb_lignes,5
4,chinook_037,Sonnet 5.5,nb_lignes,5


## 4. Conclusions
Ces résultats deviennent la **baseline** : toute variante (prompt, effort, modèle) sera comparée
à elle, sur les mêmes questions.

Premier passage : l’évaluation a d’abord mesuré le correcteur. 

Lors de la première évaluation réelle, les questions difficiles semblaient départager les modèles : 
- 50 % de réussite systématique pour Haiku 5.5, 
- 67 % pour Sonnet 5.5, avec des courbes pass^k parfaitement plates. 

Chaque question était réussie 5 fois sur 5 ou ratée 5 fois sur 5, toujours pour le même motif : un nombre de lignes différent de la référence. 
L’examen des réponses a montré que l’agent avait raison dans tous ces cas, et que les règles de notation étaient trop rigides. 

Sur les questions de classement (« Quel artiste a généré le plus de chiffre d’affaires ? »), l’agent donnait la bonne réponse mais renvoyait un top 3 ou un top 5 pour la mettre en contexte, là où la référence n’a qu’une ligne. Sur une question à résultat vide (clients ayant acheté du jazz mais jamais de rock), Haiku vérifiait après coup que les genres existaient bien, et le correcteur notait cette requête de vérification au lieu de la réponse. 

Deux règles ont été ajustées : 
- des lignes en plus à la fin d’un classement sont tolérées, 
- et le résultat noté est celui de la requête que l’agent présente dans sa réponse, retrouvée parmi les requêtes réellement exécutées. 

Chaque enregistrement conservant la trace complète de l’agent.

les résultats ont été re-notés sans nouvel appel à l’API : 
- 30/30 pour chacun des deux modèles. 

--> Deux enseignements : une métrique « exacte » encode des choix, qui doivent être confrontés aux réponses réelles avant toute conclusion ; et ces six questions, réussies de façon stable par les deux modèles, sont trop faciles pour mesurer la fiabilité. Il faudra des questions plus ambiguës pour que pass^k devienne informatif.

In [ ]:
from text_to_sql_mcp.evaluation import regrade

avant1, avant2 = eval1, eval2
eval1 = regrade(eval1, questions, references)
eval2 = regrade(eval2, questions, references)
df1, df2 = tableau(eval1), tableau(eval2)

# Ce que la re-notation a changé
changements = pd.DataFrame(
    [
        {
            "évaluation": nom_eval,
            "question": a.question_id,
            "modèle": nom(a.model),
            "avant": a.reason,
            "après": n.reason,
        }
        for nom_eval, avant, apres in (("1", avant1, eval1), ("2", avant2, eval2))
        for a, n in zip(avant, apres, strict=True)
        if a.reason != n.reason
    ]
)
print(f"{len(changements)} verdict(s) modifié(s)")
changements